# 🔐 01 · 密码学基础

> 挖矿不是玄学，是数学。SHA-256、RIPEMD-160、椭圆曲线。

## 1.1 SHA-256 — 矿工唯一的工具

**五大特性：**
- 任意输入 → 固定 256 位输出
- 确定性：同输入永远同输出
- 雪崩效应：改 1 bit 输入 → 约 50% 输出 bit 翻转
- 抗原像：给哈希值推不出原输入
- 抗碰撞：找不到两个不同输入产生相同哈希

In [ ]:
import hashlib, time, os, struct, random, binascii
from typing import Tuple, Optional

def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def double_sha256(data: bytes) -> bytes:
    # Bitcoin uses double SHA-256
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

print("OK - SHA-256 functions ready")

### 雪崩效应 — 差之毫厘，缪之千里

In [ ]:
def avalanche_demo():
    a = b"hello world"
    b = b"hello worlD"  # only changed case of last char

    h1 = sha256(a)
    h2 = sha256(b)

    print(f"  hello world  -> {h1.hex()}")
    print(f"  hello worlD  -> {h2.hex()}")

    diff = sum(1 for x, y in zip(h1, h2) if x != y)
    print(f"\n  {len(h1)} bytes = {len(h1)*8} bits total")
    print(f"  {diff*8} bits differ ({diff*8/len(h1)/8*100:.0f}% avalanche)")
    print(f"  Only {len(h1)-diff} bytes / {len(h1)} are the same")

avalanche_demo()

### 前导零 — 挖矿就是你做的工作

矿工做的事情：不断试 nonce，直到区块头的 double SHA-256 以足够多的零开头。

每多一个前导零字节（8 bits），难度 × 256。

In [ ]:
def mine(target_zeros: int):
    nonce = 0
    target = b"\x00" * target_zeros
    start = time.time()

    while True:
        guess = sha256(f"block_data|{nonce}".encode())
        if guess.startswith(target):
            break
        nonce += 1

    elapsed = time.time() - start
    print(f"  Zeros needed: {target_zeros} bytes ({target_zeros*8} bits)")
    print(f"  Tried: {nonce:,} nonces")
    print(f"  Time: {elapsed:.3f}s")
    print(f"  Hashrate: {nonce/elapsed:,.0f} H/s")
    print(f"  Found hash: {guess.hex()}")
    print()

for z in [1, 2, 3, 4]:
    mine(z)
    # Note: z=5 may take a while on pure Python!

## 1.2 比特币地址生成

```
公钥 → SHA-256 → RIPEMD-160 → 加版本字节 → 双SHA256取checksum → Base58
```

这叫 **Base58Check** 编码。做这么多层是为了：更短的地址、内建校验、支持前缀区分。

In [ ]:
def hash160(data: bytes) -> bytes:
    # Bitcoin HASH160 = RIPEMD-160(SHA-256(data))
    sha = hashlib.sha256(data).digest()
    rip = hashlib.new("ripemd160")
    rip.update(sha)
    return rip.digest()

def base58_encode(data: bytes) -> str:
    alphabet = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
    n = int.from_bytes(data, "big")
    result = []
    while n > 0:
        n, rem = divmod(n, 58)
        result.append(alphabet[rem])
    for b in data:
        if b == 0:
            result.append("1")
        else:
            break
    return "".join(reversed(result))

def pubkey_to_address(pubkey: bytes, prefix: bytes = b"\x00") -> str:
    # P2PKH address (starts with '1')
    h160 = hash160(pubkey)
    payload = prefix + h160
    checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    return base58_encode(payload + checksum)

# Demo with random "pubkey"
demo_pk = os.urandom(33)
addr = pubkey_to_address(demo_pk)
print(f"Address: {addr}")
print(f"(Starts with '1' = P2PKH, mainnet)")

## 1.3 ECDSA — 椭圆曲线数字签名

比特币使用 **secp256k1** 曲线方程：**y² = x³ + 7**

- 私钥：随机 256 位整数（保密）
- 公钥：私钥 × 生成点 G（可公开）
- 签名：用私钥对交易哈希签名
- 验证：用公钥验证签名

> 安全性基础：EC 离散对数问题（知道 G 和 k·G，算不出 k）

In [ ]:
# 极小椭圆曲线演示 (不是 secp256k1!)
# y^2 = x^3 + ax + b (mod p)

class TinyCurve:
    def __init__(self, a, b, p, Gx, Gy, n):
        self.a, self.b, self.p = a, b, p
        self.G = (Gx, Gy)
        self.n = n

    def modinv(self, k):
        return pow(k, -1, self.p)

    def point_add(self, P, Q):
        if P is None: return Q
        if Q is None: return P
        x1, y1 = P
        x2, y2 = Q
        if x1 == x2 and (y1 + y2) % self.p == 0:
            return None  # point at infinity
        if P == Q:
            lam = (3*x1*x1 + self.a) * self.modinv(2*y1) % self.p
        else:
            lam = (y2 - y1) * self.modinv(x2 - x1) % self.p
        x3 = (lam*lam - x1 - x2) % self.p
        y3 = (lam*(x1 - x3) - y1) % self.p
        return (x3, y3)

    def scalar_mult(self, k, P=None):
        if P is None: P = self.G
        result = None
        addend = P
        while k:
            if k & 1:
                result = self.point_add(result, addend)
            addend = self.point_add(addend, addend)
            k >>= 1
        return result

# y^2 = x^3 + 2x + 3 mod 97
c = TinyCurve(a=2, b=3, p=97, Gx=3, Gy=6, n=100)
sk = random.randint(1, c.n - 1)
pk = c.scalar_mult(sk)

print(f"Curve: y^2 = x^3 + {c.a}x + {c.b} mod {c.p}")
print(f"G = {c.G},  order n = {c.n}")
print(f"\nPrivate key: {sk}")
print(f"Public key:  {pk}")
print(f"Verify: k*G = {c.scalar_mult(sk)} == PK: {c.scalar_mult(sk) == pk}")
print("\n(Real secp256k1 uses p = 2^256 - 2^32 - 977, much larger!)")

## 小结

✅ SHA-256 + 双 SHA-256 是比特币的心脏
✅ 雪崩效应确保暴力破解不可行
✅ 前导零 = 工作量的度量
✅ Base58Check → 美观且防错的地址格式
✅ ECDSA → 不可伪造的数字签名

**下一课**: 把这些拼成完整的工作量证明。